In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
### Load the trained model, scaler pickle, onehot
model = load_model('model.h5')

### load encoder and scaler
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('one_hot_encoder_geography.pkl', 'rb') as f:
    onehot_encoder_geography = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [3]:
## Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [18]:
## Prepare input data by encoding categorical variables
# First, create the base dataframe
input_df = pd.DataFrame([input_data])

# Encode Gender
input_df['Gender'] = label_encoder_gender.transform(input_df[['Gender']].values)

# One-hot encode Geography
geo_encoded = onehot_encoder_geography.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geography.get_feature_names_out(['Geography']))

# Combine: drop Geography from input_df and add the one-hot encoded columns
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)

print("Prepared input_df columns:", input_df.columns.tolist())
print("input_df shape:", input_df.shape)
input_df

Prepared input_df columns: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']
input_df shape: (1, 12)


c:\Users\rashm\ANN\venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\rashm\ANN\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [5]:
input_df = pd.DataFrame([input_data])


In [6]:
input_df['Gender'] = label_encoder_gender.transform(input_df[['Gender']].values)

c:\Users\rashm\ANN\venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [7]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [10]:
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)

In [13]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0,1.0,0.0,0.0


In [20]:
## Reorder columns to match the training data structure
# The training data had: CreditScore, Gender, Age, Tenure, Balance, NumOfProducts, 
# HasCrCard, IsActiveMember, EstimatedSalary, Geography_France, Geography_Germany, Geography_Spain
expected_columns = ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
                    'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
                    'Geography_France', 'Geography_Germany', 'Geography_Spain']

# Reorder input_df to match
input_df_scaled = input_df[expected_columns]

## Scaling the input data
scaled_input = scaler.transform(input_df_scaled)
scaled_input

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [21]:
prediction = model.predict(scaled_input)
prediction

1/1 [==============================] - 0s 418ms/step


array([[0.06585598]], dtype=float32)

In [23]:
prediction_proba = prediction[0][0]

In [24]:
prediction_proba

0.06585598